# 20. Plotly Interactive Charts — `plotly`
**Python Libraries Lab · Phase 2 · Data Visualisation**

_Zoom, hover and click: interactive charts in a few lines._

✅ **Runs in Colab** as-is.

Plotly makes interactive charts. You can hover to see exact values, zoom into a busy area, and click legend items to hide them. The plotly express module makes most charts with one function call, taking a DataFrame and column names, a lot like Seaborn.

**In this notebook:** You will build a hover line chart of revenue, a bar chart split into panels by city, a sunburst that drills from category to city, and an animated chart that plays through the months. Then you will save one as an HTML file.

**How to use:** run each cell in order with **Shift + Enter**, or use **Runtime → Run all**. Read the short note above each cell first, then change the code and run it again: that is how you learn.

## Setup
The next cell makes sure the library is installed (Colab already has most of them; it takes a few seconds).

In [ ]:
!pip install -q plotly
import importlib.metadata as md
for pkg in ['plotly']:
    print(pkg, md.version(pkg))

## Step 1 · Create the ShopEasy data
Same dataset again.

In [ ]:
import numpy as np, pandas as pd

rng = np.random.default_rng(42)
n = 600
orders = pd.DataFrame({
    "order_id": np.arange(1001, 1001 + n),
    "date": pd.Timestamp("2026-01-01") + pd.to_timedelta(rng.integers(0, 90, n), unit="D"),
    "city": rng.choice(["Mumbai", "Delhi", "Pune", "Bengaluru"], n),
    "category": rng.choice(["Books", "Electronics", "Clothing", "Home"], n, p=[0.3, 0.2, 0.3, 0.2]),
    "quantity": rng.integers(1, 6, n),
    "price": rng.choice([199, 499, 999, 1499, 2999], n),
    "rating": rng.choice([1, 2, 3, 4, 5], n, p=[0.05, 0.1, 0.2, 0.35, 0.3]),
})
orders["revenue"] = orders["quantity"] * orders["price"]
orders.to_csv("shop_orders.csv", index=False)
print(orders.shape, "-> saved shop_orders.csv")
orders.head()

## Step 2 · Interactive line chart
Hover over the line to see values. Drag to zoom, double-click to reset.

In [ ]:
import plotly.express as px

daily = orders.groupby(["date", "category"], as_index=False)["revenue"].sum()
fig = px.line(daily, x="date", y="revenue", color="category", title="Daily revenue by category")
fig.show()

## Step 3 · Facets: one small chart per city
`facet_col` repeats the chart for each value.

In [ ]:
summary = orders.groupby(["city", "category"], as_index=False).agg(revenue=("revenue", "sum"), orders=("order_id", "count"))
fig = px.bar(summary, x="category", y="revenue", color="category", facet_col="city",
             hover_data=["orders"], title="Revenue by category in each city")
fig.show()

## Step 4 · Sunburst: drill down
Click a slice to zoom into it.

In [ ]:
fig = px.sunburst(orders, path=["category", "city"], values="revenue", title="Where revenue comes from")
fig.show()

## Step 5 · Animation through the months
`animation_frame` adds a play button.

In [ ]:
orders["month"] = orders["date"].dt.strftime("%Y-%m")
m = orders.groupby(["month", "city"], as_index=False).agg(revenue=("revenue", "sum"), avg_rating=("rating", "mean"), orders=("order_id", "count"))
fig = px.scatter(m, x="orders", y="revenue", size="revenue", color="city", animation_frame="month",
                 range_x=[20, 70], range_y=[0, 300_000], title="Cities month by month (press play)")
fig.show()

## Step 6 · Save and share
One HTML file that opens in any browser. Download it from the Colab Files panel.

In [ ]:
fig.write_html("cities_by_month.html")
import os
print("Saved:", os.path.getsize("cities_by_month.html") // 1024, "KB")

## ✍️ Your turn
Make a `px.histogram` of `revenue` coloured by `category`. Store the figure in `answer`.

Write your code in the next cell, run it, then run the check cell.

In [ ]:
answer = None  # your code here

In [ ]:
def check():
    ok = answer is not None and len(answer.data) == 4
    return ok, "Expected a figure with 4 traces (one per category)"

# ▶ Run this cell to check your answer
try:
    ok, msg = check()
except Exception as e:
    ok, msg = False, f"your code raised {type(e).__name__}: {e}"
print("✅ Correct! Well done." if ok else "❌ Not yet: " + msg)

### Stuck?
The next cell holds a solution. In Colab its code is hidden: double-click the cell to see it, or just run it.

In [ ]:
# @title 👀 Solution (double-click to show the code) { display-mode: "form" }
answer = px.histogram(orders, x="revenue", color="category")
answer.show()

# Check again with the solution
try:
    ok, msg = check()
except Exception as e:
    ok, msg = False, f"your code raised {type(e).__name__}: {e}"
print("✅ Correct! Well done." if ok else "❌ Not yet: " + msg)

## Recap
- **plotly.express**
- **Hover**
- **Facet**
- **Animation**
- **write_html**

**Next:** 21. Bokeh & Altair (`bokeh · altair`): [open in Colab](https://colab.research.google.com/github/aashaysocial27-maker/python-libraries-colab/blob/main/notebooks/21_bokeh_altair.ipynb)

## 💻 Run it on your own computer
1. Install Python 3.10+ from python.org (Windows: tick **Add python.exe to PATH**).
2. Open a terminal (Windows: *Command Prompt*; Mac: *Terminal*) and run:
```bash
mkdir python-lab && cd python-lab
python -m venv .venv
# Windows:  .venv\Scripts\activate      Mac/Linux:  source .venv/bin/activate
pip install plotly
pip install notebook
jupyter notebook
```
3. In Colab choose **File → Download → Download .ipynb**, then open that file in the Jupyter page that opened in your browser.

Lines that start with `!` or `%%` are notebook commands. In a plain terminal, type `!` commands without the `!`.